# 04 – Modeling & Submission (5 Wochen / Region)

**Colab:** [Open in Colab](https://colab.research.google.com/github/jspldrch/DataMining_Final-Project/blob/main/notebooks/04_modeling.ipynb) → **CPU** → Run all.

Kaggle-Format wie `sample_submission.csv`:
- **2.248 Zeilen** (eine pro Region)
- Spalten: `pred_week1` … `pred_week5` (nächste 5 Wochen nach den 91 Test-Tagen)
- Metrik: **MAE** über alle Regionen × 5 Wochen

Voraussetzung: Notebook **03** (`train_labeled.parquet`, `test_features.parquet`).

**RAM:** 03 liefert *tägliche* Labels; 04 aggregiert auf **Wochen** (`ordinal // 7`), damit Training zum 5-Wochen-Task passt und Colab nicht OOM läuft → Details in `docs/09_WEEKLY_MODELING.md`.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

REPO_URL = "https://github.com/jspldrch/DataMining_Final-Project.git"
REPO_DIR = Path("/content/DataMining_Final-Project")
RANDOM_STATE = 42
VAL_REGION_FRAC = 0.2

if IS_COLAB:
    drive.mount("/content/drive")
    if (REPO_DIR / ".git").exists():
        subprocess.run(["git", "pull", "origin", "main"], cwd=REPO_DIR, check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
        check=True,
    )
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    from scripts.colab_setup import init_environment
    env = init_environment(install_deps=False, skip_mount=True, skip_git=True)
else:
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / "scripts").exists() and (REPO_DIR.parent / "scripts").exists():
        REPO_DIR = REPO_DIR.parent
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    from scripts.colab_setup import init_environment
    env = init_environment(install_deps=True, skip_mount=True, skip_git=True)

PROJECT_ROOT = env["project_root"]
DATA_DIR = env.get("data_dir")
PROCESSED_DIR = env["processed_dir"]
SUBMISSION_DIR = env["submission_dir"]
MODE = env["mode"]

_spec = importlib.util.spec_from_file_location("dm_features", PROJECT_ROOT / "scripts" / "features.py")
_features = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_features)
feature_columns = _features.feature_columns

_spec2 = importlib.util.spec_from_file_location("weekly_model", PROJECT_ROOT / "scripts" / "weekly_model.py")
_wm = importlib.util.module_from_spec(_spec2)
_spec2.loader.exec_module(_wm)

train_df = pd.read_parquet(PROCESSED_DIR / "train_labeled.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test_features.parquet")
FEATURES = [c for c in feature_columns() if c in train_df.columns]
train_df = _wm.slim_for_modeling(train_df, FEATURES)
wk = _wm.weekly_summary(train_df)
print(
    f"Train: {wk['daily_rows']:,} Tage → {wk['weekly_rows']:,} Wochen "
    f"(~{wk['median_weeks_per_region']:.0f}/Region)"
)

SAMPLE_SUB_PATH = _wm.find_sample_submission(PROJECT_ROOT, DATA_DIR)
if SAMPLE_SUB_PATH is not None:
    SUBMISSION_TEMPLATE = SAMPLE_SUB_PATH
    print(f"Sample submission: {SAMPLE_SUB_PATH}")
else:
    SUBMISSION_TEMPLATE = _wm.build_submission_template(test_df["region_id"])
    print(
        "sample_submission.csv nicht gefunden — Template aus test_features "
        f"({len(SUBMISSION_TEMPLATE):,} Regionen). Optional: Datei nach "
        f"{DATA_DIR or PROJECT_ROOT / 'data'}/sample_submission.csv legen."
    )

print(f"Train labeled: {len(train_df):,} | Test rows: {len(test_df):,}")

## 1. Train/Valid (wöchentliche Fenster → 5 Wochen Targets)

In [ ]:
X_tr, y_tr, X_va, y_va, val_regions = _wm.build_region_holdout(
    train_df, FEATURES, val_region_frac=VAL_REGION_FRAC, seed=RANDOM_STATE
)
print(f"Train-Fenster (wöchentlich): {len(X_tr):,} | Val-Regionen: {len(val_regions)}")
print(f"y shape: train {y_tr.shape}, val {y_va.shape}  (~{len(X_tr)/wk['regions']:.0f} Fenster/Region)")

## 2. Baselines (MAE wie Kaggle)

In [ ]:
train_weekly = _wm.daily_to_weekly(train_df)
last_score = train_weekly.sort_values("ordinal").groupby("region_id")["score"].last()
region_median = train_weekly.groupby("region_id")["score"].median()

def eval_preds(y_true, y_pred, name):
    print(f"{name:32s}  MAE={_wm.mae_kaggle(y_true, y_pred):.4f}")


persist = np.column_stack([
    last_score.reindex(val_regions).fillna(0).to_numpy() for _ in range(5)
])
eval_preds(y_va, persist, "Persistence (letzter Score ×5)")

med = np.column_stack([
    region_median.reindex(val_regions).fillna(train_df["score"].median()).to_numpy()
    for _ in range(5)
])
eval_preds(y_va, med, "Regional median ×5")

## 3. LightGBM (ein Modell pro Woche)

In [ ]:
models = []
val_preds = np.zeros_like(y_va)

for week in range(5):
    m = lgb.LGBMRegressor(
        objective="regression",
        metric="mae",
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE + week,
        n_jobs=-1,
        verbose=-1,
    )
    m.fit(
        X_tr,
        y_tr[:, week],
        eval_set=[(X_va, y_va[:, week])],
        eval_metric="mae",
        callbacks=[lgb.early_stopping(40, verbose=False)],
        categorical_feature=["region_id"],
    )
    models.append(m)
    val_preds[:, week] = _wm.clip_scores(m.predict(X_va))

eval_preds(y_va, val_preds, "LightGBM (5 Modelle)")

## 4. Finales Training & Kaggle-Submission

In [ ]:
X_all, y_all, _ = _wm.build_sliding_samples(
    train_df, FEATURES, already_weekly=False
)  # weekly collapse inside (same as holdout)
final_models = []
for week in range(5):
    m = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=models[week].best_iteration_ or 600,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE + week,
        n_jobs=-1,
        verbose=-1,
    )
    m.fit(X_all, y_all[:, week], categorical_feature=["region_id"])
    final_models.append(m)

X_test = _wm.test_features_last_row(test_df, FEATURES)
test_preds = _wm.predict_week_columns(final_models, X_test)

submission = _wm.submission_frame(X_test["region_id"], test_preds)
submission = _wm.align_to_sample_submission(submission, SUBMISSION_TEMPLATE)

out_path = SUBMISSION_DIR / f"submission_{MODE}.csv"
submission.to_csv(out_path, index=False)

print(f"Gespeichert: {out_path}")
print(f"Zeilen: {len(submission):,} (erwartet 2.248)")
print(f"Spalten: {list(submission.columns)}")
submission.head(10)